# Technical Crypto Optimization

Notebook dedicado ao tuning otimizado do XGBoost para `Direction_t+1`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns

from src.data.historical_loader import load_historical_asset_dataframe
from src.data.technical_preprocessing import prepare_technical_market_dataframe
from src.features.technical_indicators import OPTIMIZED_XGB_FEATURES, add_technical_indicators, add_technical_targets
from src.models.technical_training import optimize_xgboost_technical


In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'CSvs'
ASSET_NAME = 'Binance_BTCUSDT_d'
FEATURES = list(OPTIMIZED_XGB_FEATURES)


In [ ]:
raw_asset = load_historical_asset_dataframe(DATA_DIR, ASSET_NAME)
technical_base = prepare_technical_market_dataframe(raw_asset)
technical_df = add_technical_indicators(technical_base)
technical_df = add_technical_targets(technical_df)
technical_df = technical_df.dropna()


In [ ]:
best_params, cv_results, metrics, confusion, importance_df = optimize_xgboost_technical(
    technical_df,
    FEATURES,
    target_column='Direction_t+1',
)
print(best_params)
print(metrics)
importance_df.head(20)


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (Last Fold)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importance_df.head(20))
plt.title('Top 20 Feature Importance')
plt.tight_layout()
plt.show()
